# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

# Load the standard BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# A sample sentence to demonstrate wordpiece tokenization
sample_sentence = "The quick brown fox jumps over the lazy dog and artificial intelligence."
print(f"Sample Sentence: {sample_sentence}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Sample Sentence: The quick brown fox jumps over the lazy dog and artificial intelligence.


In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | the          |  1996
    2 | quick        |  4248
    3 | brown        |  2829
    4 | fox          |  4419
    5 | jumps        | 14523
    6 | over         |  2058
    7 | the          |  1996
    8 | lazy         | 13971
    9 | dog          |  3899
   10 | and          |  1998
   11 | artificial   |  7976
   12 | intelligence |  4454
   13 | .            |  1012
   14 | [SEP]        |   102
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (14, '[SEP]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]'), (20, '[PAD]'), (21, '[PAD]

### Exercise 1 reflection
- **[CLS] and [SEP]**: The `[CLS]` (Classification) token is prepended to every sequence; its final hidden state is used as the aggregate sequence representation for classification tasks. The `[SEP]` (Separator) token marks the end of a sequence or the boundary between two sentences.
- **Attention Mask**: The attention mask is a binary tensor (1s for real tokens, 0s for padding). It tells the self-attention mechanism to ignore the `[PAD]` tokens during calculation, ensuring that irrelevant padding doesn't influence the representation of the actual text.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

# Initialize the sentiment analysis pipeline with a DistilBERT model
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Testing with a sample sentence
sentence = "This tutorial on BERT is incredibly helpful and easy to follow!"
prediction = sentiment_pipeline(sentence)
print(f"Sentence: {sentence}")
print(f"Prediction: {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence: This tutorial on BERT is incredibly helpful and easy to follow!
Prediction: [{'label': 'POSITIVE', 'score': 0.9995172023773193}]


### Exercise 2 reflection
- **Expectation**: Yes, the label matches. The sentence was clearly positive, and DistilBERT correctly identified it as 'POSITIVE'.
- **Confidence**: The score (e.g., 0.99) represents the probability assigned by the softmax layer. High confidence indicates the input closely aligns with the patterns learned during fine-tuning on the SST-2 dataset.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    """
    A custom class to perform sentiment analysis using a fine-tuned BERT model.
    """
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.max_length = max_length
        # Initialize the tokenizer for the specific model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        # Load the model with a sequence classification head
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        # Use CUDA if available for faster processing
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval() # Switch to evaluation mode (disables dropout)

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # Convert text to tokens and then to tensors
        # We apply padding and truncation to ensure consistent tensor shapes
        return self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)

    def predict(self, text: str) -> Dict[str, any]:
        # 1. Prepare the input tensors
        inputs = self.preprocess(text)
        with torch.no_grad():
            # 2. Forward pass: get raw logits from the model
            outputs = self.model(**inputs)
            # 3. Activation: use Softmax to get a probability distribution
            probs = F.softmax(outputs.logits, dim=-1)
            # 4. Selection: find the index of the highest probability
            val, index = torch.max(probs, dim=-1)

        # Map the numeric index back to the human-readable label (e.g., POSITIVE)
        label = self.model.config.id2label[index.item()]
        return {"label": label, "probability": val.item()}

In [ ]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "I am so excited to learn about deep learning today!",
    "The weather is gloomy and the traffic is making me frustrated."
]
for text in samples:
    res = analyzer.predict(text)
    print(f"Text: {text}")
    print(f"Result: {res}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I am so excited to learn about deep learning today!
Result: {'label': 'POSITIVE', 'probability': 0.9997952580451965}

Text: The weather is gloomy and the traffic is making me frustrated.
Result: {'label': 'NEGATIVE', 'probability': 0.9996761083602905}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    """
    A class to extract named entities (People, Locations, etc.) using BERT.
    """
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def recognize(self, text: str):
        # 1. Tokenize with offsets to track character positions
        inputs = self.tokenizer(text, return_offsets_mapping=True, return_tensors="pt").to(self.device)
        offsets = inputs.pop("offset_mapping")[0]

        with torch.no_grad():
            outputs = self.model(**inputs)

        # 2. Get predictions and tokens
        predictions = torch.argmax(outputs.logits, dim=2)[0]
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        entities = []
        for i, (token, pred) in enumerate(zip(tokens, predictions)):
            label = self.model.config.id2label[pred.item()]

            # 3. Filter out 'O' and special tokens (CLS/SEP)
            if label != "O" and token not in self.tokenizer.all_special_tokens:
                start, end = offsets[i]
                entities.append({
                    "text": token,
                    "entity": label,
                    "start": start.item(),
                    "end": end.item()
                })

        # SUBWORD HANDLING EXPLANATION:
        # BERT uses WordPiece tokenization. Words not in the vocabulary are split into subwords
        # starting with '##'. In this implementation, each subword piece is labeled individually.
        # To 'handle' them, one would typically merge pieces starting with '##' into the previous
        # token and adjust the 'end' offset to match the full word span.

        return entities

In [ ]:
ner = BERTNamedEntityRecognizer()
sample_text = "George Washington was the first president of the United States and lived at Mount Vernon."
entities = ner.recognize(sample_text)

print(f"Text: {sample_text}\n")
print(f"{'Token':<12} | {'Entity':<8} | {'Start':<6} | {'End':<4}")
print("-" * 40)
for ent in entities:
    print(f"{ent['text']:<12} | {ent['entity']:<8} | {ent['start']:<6} | {ent['end']:<4}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: George Washington was the first president of the United States and lived at Mount Vernon.

Token        | Entity   | Start  | End 
----------------------------------------
George       | B-PER    | 0      | 6   
Washington   | I-PER    | 7      | 17  
United       | B-LOC    | 49     | 55  
States       | I-LOC    | 56     | 62  
Mount        | B-LOC    | 76     | 81  
Vernon       | I-LOC    | 82     | 88  


## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only (Bidirectional) | Decoder-only (Autoregressive) |
| Primary purpose | Understanding context from both directions | Predicting the next token in a sequence |
| Typical use cases | NER, Sentiment Analysis, QA | Text generation, Chatbots, Creative writing |
| Strengths | Deep contextual understanding of whole sentences | Excellent at generating fluent, long-form text |
| Weaknesses | Not naturally suited for text generation | Directionality is limited (sees only previous tokens) |

## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **Encoding**: BERT serves as the 'encoder' in the retrieval stage. It transforms both the user's query and massive amounts of documents into dense vector embeddings. Because BERT is bidirectional, these embeddings capture rich semantic meaning rather than just keyword matches.

2. **Vector DB**: These embeddings are indexed in a vector database (like Pinecone or FAISS). When a user asks a question, BERT encodes the query, and the system performs a similarity search (e.g., Cosine Similarity) to find document chunks whose vectors are closest to the query vector.

3. **Generative Handoff**: The most relevant document chunks are retrieved and prepended to the user's original query as "context." This combined prompt is then fed to a generative model (like GPT), which uses the retrieved facts to synthesize an accurate answer.

4. **Application**: A concrete example is a **Technical Support Bot** for a large software company. BERT can retrieve specific troubleshooting steps from thousands of pages of internal documentation, ensuring the generative model provides an answer based on factual company manuals rather than hallucinating.